# Integrated Intelligent Systems: Tutorial 3 - Body Motion Problem (Scientific Object) (3 Pts)

This notebook introduces a formulation body motion problem, central to designing and implementing Cognition-Enabled Robot Agents, as a description logic system.

## Part 1: Robot-World Manipulation System as Scientific Object (E1–E6) (1.5 Pt)

**Title:** Formalizing the Robot Agent: From Architecture to Description Logic Reasoners

![Alternative Text](RWMS.png)

### Scenario Background
An autonomous robotic agent (the PR2 in the EASE apartment) is deployed to execute a manipulation task ("Serve Eva a drink"). To validate this agent scientifically, we must formalize the system as a **Robot-World Manipulation System (RWMS)**, separate the controlling subsystem from the controlled body-world system, define trajectories as trees, and assess runs against an external problem standard using a description logic reasoner.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E1 · Architectural Cut & Subsystem Separation**
   - In the graphical representation of the RWMS, a strict architectural "cut" is placed between the manipulation executive and the body-world system ($[B \bowtie W]$). Explain the nature of this cut: why is the manipulation executive classified as a `dul:Agent` (non-physical program) while the body-world system is classified under physical objects, and how do they interact exclusively across the boundary?

2. **E2 · Forward-Branching Time & World Choices**
   - During an operation on a fridge door or container seal, the same action token (e.g., a position-controlled pull) can lead to completely divergent physical outcomes depending on world choices (e.g., seal releases vs. seal holds tight). Draw or describe the structure of the resulting courses, and explain why the possible courses form a tree that branches *only forward* from the initial state $x_0$.

3. **E3 · Trajectory Classification & Priority Hierarchies**
   - The problem standard $P$ sorts every physical trajectory into one of four classes: **SOL** (Solution), **REPD** (Repaired), **NON** (Nonachievement), and **PROH** (Prohibited). Consider two runs that both successfully place the drink within Eva's reach, but where Run A tears off the container handle (`P4` violation) while Run B maintains licensed contact. Classify both runs and explain why the classifier grades the *entire course* rather than just the final end-state.

4. **E4 · Description Logic Partonomy & DUL Constraints**
   - When formalizing the RWMS in description logic under DOLCE Ultra Lite (DUL), we define an RWMS as having *exactly one* executive and *exactly one* body-world system (`RWMS = (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)`). Using the transitivity of `dul:isPartOf`, explain how a reasoner (such as HermiT) automatically entails that the PR2's head camera is a component of the RWMS without explicitly hardcoding every part-whole link.

5. **E5 · Ontological Inconsistency & Entailment Limits**
   - An inconsistency in description logic serves as a strict analytical tool. Formulate a competency question that results in an ontological inconsistency when attempting to classify the *manipulation problem itself* as a physical component inside the RWMS. Furthermore, explain why numerical physical laws (such as Coulomb friction $2 \cdot \mu \cdot F \ge m \cdot (g + a)$) must be handled outside description logic by the semantic digital twin's causal model.

6. **E6 · Strength vs. Truth in Agent Validation**
   - The design thesis dictates that an agent decides within its believed causality, which can be tested for logical *strength* via competency queries and for *truth* via physical experiments. Compare how an axiom set that lacks a friction law ($\text{K}_0$) performs on competency queries versus an axiomatization containing explicit dynamical laws ($\text{K}_1/\text{K}_2$), and state what kind of test is required to validate the actual physical truth of those beliefs.


Part 1 — Robot-World Manipulation System as Scientific Object (E1–E6)

**E1 ANS** : 
The architectural cut separates the manipulation executive from the Body-World System \([B \bowtie W]\).
The manipulation executive is classified as a dul:Agent because it is the software subsystem responsible for reasoning, planning, decision-making, and controlling the robot. It is not a physical object.
The Body-World System consists of the physical robot body and its environment. For example, the PR2 robot, its cameras, gripper, fridge, and other objects in the apartment belong to the physical system.
The two subsystems interact across the architectural boundary through commands and observations. The manipulation executive sends motion commands to the physical robot and receives sensor information and feedback from it.
Therefore, the executive controls and reasons about the physical system, but it is not itself part of the physical robot.

**E2 ANS:**
Take a "pull the fridge door" action token a at the start state x₀. The world then has a choice, so one action leads to several successors:

                         x0  (door closed)
                          │  action a: position-controlled pull
              ┌───────────┴────────────┐
        [world: seal releases]   [world: seal holds]
              │                         │
        x1a (door opening)        x1b (door stuck)
              │                         │
        agent choice:             agent choice:
        continue / re-grasp       pull harder / re-grasp / give up
           ┌──┴──┐                   ┌──┴──┐
          ...   ...                 ...   ...

The tree has alternating layers. The agent picks an action, then the world picks an outcome, and so on.It branches only forward because every state has exactly one history, the path back to x₀. Two different pasts can't merge into the same node, since a node is identified by how it was reached. The past is fixed and the future is open, because the agent has several possible actions and the world has several possible responses. The set of possible courses is therefore a tree rooted at x₀ with edges pointing only forward in time.

**E3 ANS:** 
Classification of the two runs.
Run What happens Class Run A The drink ends up within Eva's reach, but the container handle is torn off (P4 violation). PROH (Prohibited) Run B The drink ends up within Eva's reach, and contact with the container stays licensed throughout. SOL (Solution)

Why Run A is PROH. A prohibition is a constraint on what the run may do along the way. Tearing off the handle breaks it, so the run is prohibited no matter how good the final state is. In the priority hierarchy, a violated prohibition ranks above goal achievement. Reaching the goal does not make up for it.
Why Run B is SOL. It achieves the goal and violates no prohibition at any point in the course, so it is a clean solution.
A note on REPD. A run would be REPD (Repaired) only if a violation occurred but was later undone and the goal was still reached. Tearing off a handle is a physical, irreversible change, so Run A cannot be repaired and stays PROH.
Why the classifier grades the entire course, not just the final end-state.
Violations happen during the run. Whether contact was licensed, or whether something was broken, is a property of the trajectory over time. The end state alone does not show it. End-state grading would hide the difference. Run A and Run B end in the same state (drink within reach), so a classifier that looked only at the end would give them the same class. That would wrongly accept a destructive run. The problem standard is defined over courses. It sorts every physical trajectory into SOL, REPD, NON or PROH, so the input to the classifier is the whole path through the tree from E2, not its last node.

**E4 ANS:**
In DUL, isPartOf is transitive, hasComponent is a sub-property of hasPart, and isComponentOf is a sub-property of isPartOf. We only assert the direct links:

head_camera isPartOf PR2_head
PR2_head isPartOf PR2
PR2 isComponentOf BodyWorldSystem
BodyWorldSystem isComponentOf RWMS

Because of transitivity, HermiT entails head_camera isPartOf RWMS automatically when it classifies the ontology. Nobody has to write that axiom by hand, and every deeper component (a screw in the camera, say) is covered the same way. The exact-one constraint (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem) is put on hasComponent, which is a simple role. OWL DL doesn't allow cardinality restrictions on transitive roles, so putting it on isPartOf would break decidability.

**E5 ANS:**
Competency question: "Can the manipulation problem P be a physical component of the RWMS?"

To formalise it, we declare ManipulationProblem ⊑ dul:Description (an abstract entity) and DisjointClasses(dul:Description, dul:PhysicalObject). We define PhysicalComponentOfRWMS ≡ dul:PhysicalObject ⊓ ∃isComponentOf.RWMS. Then we assert an individual bmp1 : ManipulationProblem ⊓ PhysicalComponentOfRWMS. The individual would have to be both abstract and physical, which are disjoint, so HermiT reports the ontology as inconsistent. This is useful because the inconsistency shows that the modelling is wrong: a problem specification is something about the system, not part of its hardware.

On numerical laws: description logics reason over classes, roles and a few datatypes. They can't do general arithmetic across several variable values, so they can't evaluate something like 2·μ·F ≥ m·(g + a) for particular numbers, let alone with continuous dynamics. That belongs in the semantic digital twin's causal/physics model (a physics simulator or numerical solver). The DL layer then receives only the symbolic result, such as slipsUnder(grasp1, carton).

**E6 ANS:** 
K₀ (no friction law): competency queries about grasp success can't be answered. Under the open-world assumption, neither "the carton holds" nor "the carton slips" is entailed. K₀ can still answer structural questions (which motion opens the fridge, what is part of what), but it is weak. It can't tell a safe grasp from an unsafe one.
K₁/K₂ (explicit dynamical laws): these can entail concrete consequences, for example that a given grasp force is too low, or which force range is enough. They are stronger because they answer more competency queries, and they let the agent reject bad plans before acting.

Strength is not truth, though. A strong axiomatization can still be wrong, for example if the friction coefficient for a wet carton is off. Logical tests like competency queries only show what the knowledge base entails. To validate that the beliefs are true, we need physical experiments: run the grasp on the real robot (or a validated simulation), compare predicted and observed outcomes (slip or no slip) over repeated trials under varied conditions, and revise the axioms when the predictions fail.


## Part 2: Robot Motion Problem and Description Logics (1.5 Pt)

**Title:** Formulating and Querying the Robot Motion Problem with Knowledge Bases & Description Logics

![Alternative Text](PR2.png)

### Scenario Background
Using the interactive Jupyter notebook and RViz simulation environment (`bmp.rviz`, `setup_demo()`), the PR2 mobile manipulator interacts with the EASE apartment model (including the fridge and various kitchen drawers). Instead of hardcoding trajectories, the robotic executive queries a knowledge base (represented via Description Logics / Prolog reasoning) to link task requests to state changes, causes (motions), and capabilities.

[Go to the programming Lab](https://binder.intel4coro.de/v2/gh/maltehue/giskard-examples/bmp_vrb?urlpath=/lab/tree/notebooks/bodyMotionProblem.ipynb)
---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E7 · Knowledge Base Grounding & Task Requests**
   - Execute the initial query `taskRequest(Request, StateChange).`Examine the resulting solution bindings (e.g., `{'Request': 'open_fridge', 'StateChange': 'open(fridge)'}` and various drawer operations). Explain how mapping abstract task requests to discrete state changes forms the top-level specification layer for the manipulation executive.

2. **E8 · Extending Queries for Causal Motions & Parameters**
   - Extend the query to incorporate motion causes: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).` Explain how the reasoner bridges the gap between a desired symbolic state change and continuous motion parameters (such as `envJointGoal`, target joint angles, and handle identifiers).

3. **E9 · Integrating Robot Capabilities (`canPerform`)**
   - Finalize the query by adding the robot capability constraint: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).`Describe what role the `Robot` variable (`'pr2'`) plays in filtering feasible trajectories through the reasoner.

4. **E10 · Specializing Request Scope (`open_fridge` vs. `open_anything`)**
   - Modify the query request parameter from general categories like `open_anything` to specifically target `open_fridge`. Explain how refining the scope in the knowledge base reduces the solution search tree and prevents unauthorized or unintended state changes in the kitchen environment.

5. **E11 · Visualization and Simulation Linkage in RViz**
   - With the background shell command executing `rviz -d .../bmp.rviz` and displaying the PR2 interacting with the fridge handle, explain how the motion parameters derived from the reasoner's output (`[0.8, b'fridge_door_joint', b'fridge_door_handle']`) are executed as physical joint trajectories in the simulation.

6. **E12 · Ontological Limits & Missing Dynamics**
   - While the knowledge base successfully infers *which* motion achieves a state change, explain why a pure Description Logic or symbolic Prolog reasoner cannot verify whether a specific grasp force will cause a wet carton to slip, and identify which subsystem must handle those physical calculations.
  

SOLUTIONS:

E7 · Knowledge Base Grounding & Task Requests

The query taskRequest(Request, StateChange) retrieves all mappings between abstract task requests and their corresponding symbolic state changes from the knowledge base. The results show that a specific request such as open_fridge maps directly to open(fridge), whereas general requests such as open_anything and close_anything map to multiple possible objects, including the fridge and kitchen drawers. This relation provides the high-level specification layer of the robot executive by defining the desired environmental state change independently of the physical actions needed to achieve it.

E8 · Extending Queries for Causal Motions & Parameters

By extending the query with causes(Motion, StateChange, MotionParam), the knowledge base connects each desired symbolic state change to a corresponding motion and its parameters. The results show that envJointGoal is used to achieve the requested changes by specifying a target joint position, the relevant environment joint, and its handle. For example, opening the fridge is associated with the parameters [0.8, fridge_door_joint, fridge_door_handle], whereas opening a drawer uses its corresponding joint, handle, and target position. Thus, the reasoner translates high-level state changes into parameterised motion specifications.

E9 · Integrating Robot Capabilities (canPerform)

By combining taskRequest, causes, and canPerform, the query identifies a suitable body motion for the PR2 robot. Restricting the request to open_fridge produces the state change open(fridge), the motion envJointGoal, and the parameters [0.8, fridge_door_joint, fridge_door_handle]. The capability relation confirms that the PR2 can execute this motion with the specified parameters. This demonstrates how the knowledge base connects an abstract task request to a robot-specific executable motion.

E10 · Specializing Request Scope (open_fridge vs. open_anything)

Specialising a general request such as open_anything to open_fridge restricts the reasoning process to the intended object. Instead of considering multiple possible state changes involving drawers, the oven drawer, and the fridge, the knowledge base returns only open(fridge). This reduces the search space and helps prevent unintended or unauthorised interactions with other objects in the environment.

E11 · Visualization and Simulation Linkage in RViz

The reasoner's output specifies the envJointGoal motion with parameters [0.8, fridge_door_joint, fridge_door_handle]. These parameters identify the target position of the fridge door joint and the corresponding handle. The motion execution system uses this specification to generate and execute the required joint trajectory in the simulation. The PR2 interacts with the fridge handle, causing the door to open, while RViz visualises the resulting physical movement. This demonstrates the connection between symbolic reasoning and physical robot execution.

E12 · Ontological Limits & Missing Dynamics

A Description Logic or Prolog-based reasoner operates on symbolic facts, relationships and logical rules. It can infer which motion achieves a desired state change, but it cannot independently calculate physical quantities such as friction, contact forces or grasp stability. Therefore, it cannot determine whether a wet carton will slip from the robot's gripper under a particular grasp force. These physical calculations must be handled by the Simulation and Digital Twin (SDT), which models the physical properties and dynamics of the environment. These physical calculations must be handled by the Semantic Digital Twin's (SDT) causal or physics model, which models physical properties, contact interactions and environmental dynamics. The symbolic knowledge base can then use qualitative outcomes, such as whether a grasp holds or slips.

# Submission guidelines

- Deadline: 24.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (If not yet, create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put the textual answers in notebook and all related media (images, rules, etc) should then be put into a folder called **Assignment3**. 